# P2 G2 정시입결 시각화 EDA

`final/final_data/P2_G2_정시입결.csv`를 단일 원천으로 사용해 입결 반영 상태, 결측, 성적/취업/진학 변수와의 관계를 점검한다.

In [1]:

from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

BASE_DIR = Path("/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2")
SOURCE_PATH = BASE_DIR / "final/final_data/P2_G2_정시입결.csv"
OUT_DIR = BASE_DIR / "final/eda/P2_G2_정시입결_visual_eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def configure_korean_font() -> str:
    preferred = ["NanumGothic", "Noto Sans CJK KR", "Noto Sans KR", "Malgun Gothic", "AppleGothic"]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in preferred:
        if name in available:
            plt.rcParams["font.family"] = name
            plt.rcParams["axes.unicode_minus"] = False
            return name
    plt.rcParams["axes.unicode_minus"] = False
    return "DejaVu Sans"


FONT_USED = configure_korean_font()
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def save_csv(df: pd.DataFrame, filename: str) -> Path:
    path = OUT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def save_fig(fig: plt.Figure, filename: str) -> Path:
    path = OUT_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    return path


def shorten(value: object, n: int = 16) -> str:
    text = str(value)
    return text if len(text) <= n else text[: n - 1] + "…"


def add_fit_line(ax, x: pd.Series, y: pd.Series) -> None:
    ok = x.notna() & y.notna()
    if ok.sum() < 10 or x[ok].nunique() < 2:
        return
    coef = np.polyfit(x[ok], y[ok], deg=1)
    xs = np.linspace(float(x[ok].min()), float(x[ok].max()), 100)
    ax.plot(xs, coef[0] * xs + coef[1], color="#374151", linewidth=1.5, alpha=0.85)


def safe_boxplot(ax, values: list[pd.Series], labels: list[str], **kwargs) -> None:
    try:
        ax.boxplot(values, tick_labels=labels, **kwargs)
    except TypeError:
        ax.boxplot(values, labels=labels, **kwargs)


df = pd.read_csv(SOURCE_PATH, low_memory=False)
source_shape = df.shape

numeric_cols = [
    "입결_프록시", "A비율", "CD비율", "F비율", "전체취업률", "건보가입취업률",
    "전체진학률", "전문대학진학률", "대학진학률", "대학원진학률", "국내진학률", "국외진학률",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["입결_존재"] = df["입결_프록시"].notna()
df["A_minus_CD"] = df["A비율"] - df["CD비율"]
df["취업_건보_gap"] = df["전체취업률"] - df["건보가입취업률"]
work_shape = df.shape

admission = df[df["입결_존재"]].copy()
admission["입결_구간"] = pd.qcut(
    admission["입결_프록시"],
    q=4,
    labels=["Q1 낮음", "Q2", "Q3", "Q4 높음"],
    duplicates="drop",
)

overview = pd.DataFrame([
    {"metric": "source_file", "value": str(SOURCE_PATH), "note": "이번 EDA의 단일 원본 CSV"},
    {"metric": "rows", "value": source_shape[0], "note": "2024 대학-학과_계열 단위"},
    {"metric": "source_columns", "value": source_shape[1], "note": "원본 CSV 컬럼 수"},
    {"metric": "working_columns", "value": work_shape[1], "note": "EDA 파생 컬럼 포함"},
    {"metric": "universities", "value": df["학교명"].nunique(), "note": "전체 대학 수"},
    {"metric": "department_labels", "value": df["학과_계열"].nunique(), "note": "정규화 학과 라벨 수"},
    {"metric": "admission_non_null", "value": int(df["입결_존재"].sum()), "note": "입결_프록시 반영 행"},
    {"metric": "admission_missing", "value": int(df["입결_프록시"].isna().sum()), "note": "입결_프록시 결측 행"},
    {"metric": "admission_covered_universities", "value": admission["학교명"].nunique(), "note": "입결 1개 이상 대학"},
    {"metric": "duplicate_key_rows", "value": int(df.duplicated(["기준연도", "학교명", "학과_계열"]).sum()), "note": "중복 기준연도+학교명+학과_계열"},
    {"metric": "font_used", "value": FONT_USED, "note": "matplotlib 한글 폰트"},
])

missing_profile = df.isna().sum().rename("missing_n").reset_index().rename(columns={"index": "column"})
missing_profile["missing_pct"] = missing_profile["missing_n"] / len(df) * 100
missing_profile["dtype"] = [str(df[col].dtype) for col in missing_profile["column"]]

numeric_stats = (
    df[numeric_cols + ["A_minus_CD", "취업_건보_gap"]]
    .describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])
    .T.reset_index()
    .rename(columns={"index": "column"})
)

feature_cols = [c for c in numeric_cols if c != "입결_프록시"] + ["A_minus_CD", "취업_건보_gap"]
corr_rows = []
for col in feature_cols:
    sub = df[["입결_프록시", col]].dropna()
    corr_rows.append({
        "feature": col,
        "n_pair": len(sub),
        "pearson": sub["입결_프록시"].corr(sub[col], method="pearson") if len(sub) >= 3 else np.nan,
        "spearman": sub["입결_프록시"].corr(sub[col], method="spearman") if len(sub) >= 3 else np.nan,
    })
correlations = pd.DataFrame(corr_rows)
correlations["abs_spearman"] = correlations["spearman"].abs()
correlations = correlations.sort_values("abs_spearman", ascending=False)

school_summary = (
    df.groupby("학교명", dropna=False)
    .agg(
        rows=("학과_계열", "size"),
        admission_n=("입결_존재", "sum"),
        admission_mean=("입결_프록시", "mean"),
        admission_median=("입결_프록시", "median"),
        A_mean=("A비율", "mean"),
        CD_mean=("CD비율", "mean"),
        F_mean=("F비율", "mean"),
        employment_mean=("전체취업률", "mean"),
        advancement_mean=("전체진학률", "mean"),
    )
    .reset_index()
)
school_summary["admission_row_coverage_pct"] = school_summary["admission_n"] / school_summary["rows"] * 100
school_summary["입결커버구간"] = pd.cut(
    school_summary["admission_row_coverage_pct"],
    bins=[-0.1, 0, 25, 50, 75, 100],
    labels=["0%", "0~25%", "25~50%", "50~75%", "75~100%"],
)

department_signal = (
    admission.groupby("학과_계열", dropna=False)
    .agg(
        n=("입결_프록시", "size"),
        universities=("학교명", "nunique"),
        admission_median=("입결_프록시", "median"),
        admission_mean=("입결_프록시", "mean"),
        A_median=("A비율", "median"),
        CD_median=("CD비율", "median"),
        employment_median=("전체취업률", "median"),
        advancement_median=("전체진학률", "median"),
    )
    .reset_index()
)
department_signal["lift_vs_overall_median"] = department_signal["admission_median"] - admission["입결_프록시"].median()
department_signal = department_signal.sort_values(["admission_median", "n"], ascending=[False, False])

policy_summary = (
    df.groupby("학점포기제유무", dropna=False)
    .agg(
        rows=("학과_계열", "size"),
        admission_n=("입결_존재", "sum"),
        admission_median=("입결_프록시", "median"),
        admission_mean=("입결_프록시", "mean"),
        A_mean=("A비율", "mean"),
        CD_mean=("CD비율", "mean"),
        employment_mean=("전체취업률", "mean"),
        advancement_mean=("전체진학률", "mean"),
    )
    .reset_index()
)
policy_summary["admission_coverage_pct"] = policy_summary["admission_n"] / policy_summary["rows"] * 100

missing_universities = (
    school_summary[school_summary["admission_n"].eq(0)]
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

top_bottom_examples = pd.concat(
    [
        admission.nlargest(30, "입결_프록시").assign(example_group="top_30"),
        admission.nsmallest(30, "입결_프록시").assign(example_group="bottom_30"),
    ],
    ignore_index=True,
)

save_csv(overview, "00_source_contract.csv")
save_csv(missing_profile, "01_missing_profile.csv")
save_csv(numeric_stats, "02_numeric_descriptive_stats.csv")
save_csv(correlations, "03_correlations_with_admission.csv")
save_csv(school_summary.sort_values(["admission_row_coverage_pct", "admission_n"], ascending=[False, False]), "04_university_summary.csv")
save_csv(department_signal, "05_department_signal.csv")
save_csv(policy_summary, "06_policy_summary.csv")
save_csv(missing_universities, "07_zero_admission_universities.csv")
save_csv(top_bottom_examples, "08_top_bottom_admission_examples.csv")

display(Markdown(f"""
## P2 G2 정시입결 CSV EDA

- 원본 CSV: `{SOURCE_PATH}`
- 원본 행/열: `{source_shape[0]:,}` x `{source_shape[1]:,}`
- 입결 반영 행: `{int(df["입결_존재"].sum()):,}` / `{len(df):,}` (`{df["입결_존재"].mean()*100:.2f}%`)
- 입결 커버 대학: `{admission["학교명"].nunique():,}` / `{df["학교명"].nunique():,}`
- 입결 결측 행: `{int(df["입결_프록시"].isna().sum()):,}` (`{df["입결_프록시"].isna().mean()*100:.2f}%`)
- 중복 키: `{int(df.duplicated(["기준연도", "학교명", "학과_계열"]).sum()):,}`건
- 한글 폰트: `{FONT_USED}`
"""))

display(overview)
display(Markdown("### 핵심 결측 프로필"))
display(missing_profile[missing_profile["missing_n"].gt(0)].sort_values("missing_pct", ascending=False))
display(Markdown("### 입결과 숫자형 변수 상관"))
display(correlations)
display(Markdown("### 입결 0개 대학 상위"))
display(missing_universities[["학교명", "rows", "admission_n", "admission_row_coverage_pct"]].head(20))


# Figure 01: target distribution and missing contract
fig, axes = plt.subplots(2, 2, figsize=(15.5, 10), constrained_layout=True)
ax = axes[0, 0]
ax.hist(admission["입결_프록시"], bins=35, color="#2563EB", alpha=0.82, edgecolor="white")
median = admission["입결_프록시"].median()
mean = admission["입결_프록시"].mean()
ax.axvline(median, color="#DC2626", linestyle="--", linewidth=1.5, label=f"중앙값 {median:.2f}")
ax.axvline(mean, color="#111827", linestyle=":", linewidth=1.5, label=f"평균 {mean:.2f}")
ax.set_title("입결_프록시 분포")
ax.set_xlabel("입결_프록시")
ax.set_ylabel("대학-학과 수")
ax.legend(frameon=False)

ax = axes[0, 1]
q = admission["입결_프록시"].quantile([.01, .05, .25, .5, .75, .95, .99]).rename("value").reset_index()
q["index"] = q["index"].map(lambda v: f"p{int(v*100):02d}")
ax.axis("off")
tbl = ax.table(
    cellText=[[idx, f"{val:.2f}"] for idx, val in zip(q["index"], q["value"])],
    colLabels=["분위", "입결_프록시"],
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.05, 1.35)
ax.set_title("입결 분위수")

ax = axes[1, 0]
miss_view = missing_profile[missing_profile["missing_n"].gt(0)].sort_values("missing_pct", ascending=True)
ax.barh(miss_view["column"].map(shorten), miss_view["missing_pct"], color="#F97316", alpha=0.84)
ax.set_title("컬럼별 결측률")
ax.set_xlabel("missing %")

ax = axes[1, 1]
coverage_counts = school_summary["입결커버구간"].value_counts().sort_index()
ax.bar(coverage_counts.index.astype(str), coverage_counts.values, color="#0EA5E9", alpha=0.82)
ax.set_title("대학별 입결 행 커버리지 구간")
ax.set_xlabel("입결 행 커버리지")
ax.set_ylabel("대학 수")
save_fig(fig, "fig01_target_missing_contract.png")
plt.show()


# Figure 02: feature signal
fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
ax = axes[0]
corr_view = correlations.dropna(subset=["spearman"]).sort_values("spearman")
colors = np.where(corr_view["spearman"].ge(0), "#2563EB", "#DC2626")
ax.barh(corr_view["feature"].map(shorten), corr_view["spearman"], color=colors, alpha=0.84)
ax.axvline(0, color="#374151", linewidth=1)
ax.set_title("입결_프록시와 Spearman 상관")
ax.set_xlabel("Spearman r")

ax = axes[1]
dept_top = department_signal[department_signal["n"].ge(10)].head(15).sort_values("admission_median")
ax.barh(dept_top["학과_계열"].map(shorten), dept_top["admission_median"], color="#7C3AED", alpha=0.82)
ax.axvline(admission["입결_프록시"].median(), color="#374151", linestyle="--", linewidth=1.1, label="전체 중앙값")
ax.set_title("입결 중앙값 상위 학과_계열 (n>=10)")
ax.set_xlabel("입결 중앙값")
ax.legend(frameon=False, loc="lower right")
save_fig(fig, "fig02_feature_signal_rank.png")
plt.show()


# Figure 03: scatter matrix with manually checked spacing
fig, axes = plt.subplots(2, 2, figsize=(17, 12), constrained_layout=True)
scatter_specs = [
    ("A비율", "입결 vs A비율"),
    ("CD비율", "입결 vs CD비율"),
    ("전체취업률", "입결 vs 전체취업률"),
    ("전체진학률", "입결 vs 전체진학률"),
]
palette = {"O": "#2563EB", "X": "#F97316"}
for ax, (xcol, title) in zip(axes.ravel(), scatter_specs):
    sub = admission[[xcol, "입결_프록시", "학점포기제유무"]].dropna()
    colors = sub["학점포기제유무"].map(palette).fillna("#6B7280")
    ax.scatter(sub[xcol], sub["입결_프록시"], s=18, c=colors, alpha=0.42, edgecolor="none")
    add_fit_line(ax, sub[xcol], sub["입결_프록시"])
    ax.set_title(title)
    ax.set_xlabel(xcol)
    ax.set_ylabel("입결_프록시")
handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#2563EB", label="O", markersize=8),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#F97316", label="X", markersize=8),
]
fig.legend(handles=handles, title="학점포기제", loc="upper right", bbox_to_anchor=(0.995, 0.995), frameon=True)
save_fig(fig, "fig03_admission_scatter_checked.png")
plt.show()


# Figure 04: university coverage diagnostics
fig, axes = plt.subplots(1, 2, figsize=(17, 8.5), constrained_layout=True)
ax = axes[0]
cov_plot = school_summary.copy()
size = 18 + np.sqrt(cov_plot["rows"].clip(lower=1)) * 18
ax.scatter(
    cov_plot["rows"],
    cov_plot["admission_row_coverage_pct"],
    s=size,
    c=np.where(cov_plot["admission_n"].eq(0), "#DC2626", "#2563EB"),
    alpha=0.62,
    edgecolor="white",
    linewidth=0.5,
)
ax.set_title("대학별 학과 수 vs 입결 행 커버리지")
ax.set_xlabel("대학 내 학과_계열 행 수")
ax.set_ylabel("입결 반영 행 비율 %")
label_rows = cov_plot.sort_values("rows", ascending=False).head(7)
for _, row in label_rows.iterrows():
    ax.annotate(shorten(row["학교명"], 12), (row["rows"], row["admission_row_coverage_pct"]), fontsize=8, xytext=(4, 4), textcoords="offset points")

ax = axes[1]
zero_top = missing_universities.head(18).sort_values("rows")
ax.barh(zero_top["학교명"].map(shorten), zero_top["rows"], color="#DC2626", alpha=0.78)
ax.set_title("입결 0개 대학 중 행 수 상위")
ax.set_xlabel("대학 내 학과_계열 행 수")
save_fig(fig, "fig04_university_coverage_checked.png")
plt.show()


# Figure 05: group diagnostics
fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
ax = axes[0, 0]
policy_plot = policy_summary.sort_values("admission_median")
ax.bar(policy_plot["학점포기제유무"], policy_plot["admission_median"], color=[palette.get(v, "#6B7280") for v in policy_plot["학점포기제유무"]], alpha=0.82)
for i, row in policy_plot.reset_index(drop=True).iterrows():
    y = row["admission_median"]
    ax.text(i, y + 1, f"n={int(row['admission_n'])}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, min(105, max(100, policy_plot["admission_median"].max() + 8)))
ax.set_title("학점포기제 유무별 입결 중앙값")
ax.set_xlabel("학점포기제유무")
ax.set_ylabel("입결 중앙값")

ax = axes[0, 1]
box_values = [admission.loc[admission["학점포기제유무"].eq(v), "입결_프록시"].dropna() for v in ["O", "X"] if (admission["학점포기제유무"].eq(v)).any()]
box_labels = [v for v in ["O", "X"] if (admission["학점포기제유무"].eq(v)).any()]
safe_boxplot(ax, box_values, box_labels, patch_artist=True, showfliers=False)
ax.set_title("학점포기제 유무별 입결 박스플롯")
ax.set_xlabel("학점포기제유무")
ax.set_ylabel("입결_프록시")

ax = axes[1, 0]
top_n_dept = department_signal.nlargest(15, "n").sort_values("n")
ax.barh(top_n_dept["학과_계열"].map(shorten), top_n_dept["n"], color="#475569", alpha=0.82)
ax.set_title("입결 반영 행이 많은 학과_계열")
ax.set_xlabel("입결 반영 행 수")

ax = axes[1, 1]
top_school = school_summary[school_summary["admission_n"].ge(5)].nlargest(12, "admission_median").sort_values("admission_median")
ax.barh(top_school["학교명"].map(shorten), top_school["admission_median"], color="#16A34A", alpha=0.82)
ax.set_title("입결 중앙값 상위 대학 (입결 n>=5)")
ax.set_xlabel("입결 중앙값")
save_fig(fig, "fig05_group_diagnostics_checked.png")
plt.show()


top_corr = correlations.dropna(subset=["spearman"]).sort_values("spearman", ascending=False).head(4)
neg_corr = correlations.dropna(subset=["spearman"]).sort_values("spearman").head(3)
top_depts = department_signal[department_signal["n"].ge(10)].head(8)
zero_n = len(missing_universities)

interpretation = f"""
## 해석

**Target 설계:**  
`입결_프록시`를 target으로 둔다. 이 파일은 `{SOURCE_PATH.name}` 단일 CSV에서 온 2024년 대학-학과_계열 단위 데이터이며, `입결_프록시`는 정시 입결 70% cut 또는 정규화된 입결 프록시다.

**관찰:**  
- 입결 반영 행은 {int(df["입결_존재"].sum()):,}개, 커버 대학은 {admission["학교명"].nunique():,}개다.
- 입결 결측은 {int(df["입결_프록시"].isna().sum()):,}개({df["입결_프록시"].isna().mean()*100:.2f}%)다.
- 입결 0개 대학은 {zero_n:,}개다. 이 파일만 보면 원인 분해는 불가능하고, 현상은 대학 단위로 입결 반영 행이 0개인 상태로 확인된다.
- 상관이 큰 변수: {", ".join(top_corr["feature"].tolist())}
- 음의 방향 변수: {", ".join(neg_corr["feature"].tolist())}
- 입결 중앙값 상위 학과군(n>=10): {", ".join(top_depts["학과_계열"].astype(str).head(6).tolist())}

**원인 가설:**  
입결이 높은 행은 진학률과 A비율이 함께 높은 경향이 있고, CD비율은 반대로 움직인다. 다만 이는 학과 구성효과와 대학 단위 선발 구조가 섞인 상관이며 인과로 해석하면 안 된다.

**제한:**  
입결 결측은 무작위 결측이 아니다. 크롤/파싱/학과 매칭 범위에서 생긴 구조적 결측이므로 모델링할 때는 `입결_프록시.notna()`를 명시적으로 필터링해야 한다.

**결론:**  
이 CSV는 입결이 반영된 3,652개 대학-학과 행을 대상으로 성적분포·취업률·진학률과의 탐색 분석에 사용할 수 있다. 다음 개선은 입결 0개 대학과 낮은 커버리지 대학을 줄이는 방향이어야 한다.
"""
display(Markdown(interpretation))

print("saved_dir=", OUT_DIR)
for path in sorted(OUT_DIR.glob("*")):
    print(path.name, path.stat().st_size)



## P2 G2 정시입결 CSV EDA

- 원본 CSV: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/final_data/P2_G2_정시입결.csv`
- 원본 행/열: `10,242` x `17`
- 입결 반영 행: `3,652` / `10,242` (`35.66%`)
- 입결 커버 대학: `155` / `227`
- 입결 결측 행: `6,590` (`64.34%`)
- 중복 키: `0`건
- 한글 폰트: `NanumGothic`


,metric,value,note
0,source_file,/home/sieg/projects-wsl/SBS_dataScience/workbo...,이번 EDA의 단일 원본 CSV
1,rows,10242,2024 대학-학과_계열 단위
2,source_columns,17,원본 CSV 컬럼 수
3,working_columns,20,EDA 파생 컬럼 포함
4,universities,227,전체 대학 수
5,department_labels,4195,정규화 학과 라벨 수
6,admission_non_null,3652,입결_프록시 반영 행
7,admission_missing,6590,입결_프록시 결측 행
8,admission_covered_universities,155,입결 1개 이상 대학
9,duplicate_key_rows,0,중복 기준연도+학교명+학과_계열


### 핵심 결측 프로필

,column,missing_n,missing_pct,dtype
4,입결_프록시,6590,64.342902,float64
8,전체취업률,2765,26.996680,float64
9,건보가입취업률,2765,26.996680,float64
19,취업_건보_gap,2765,26.996680,float64
11,전문대학진학률,2655,25.922671,float64
10,전체진학률,2655,25.922671,float64
12,대학진학률,2655,25.922671,float64
13,대학원진학률,2655,25.922671,float64
14,국내진학률,2655,25.922671,float64
15,국외진학률,2655,25.922671,float64


### 입결과 숫자형 변수 상관

...

/tmp/ipykernel_144851/749975098.py:325: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_144851/749975098.py:355: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_144851/749975098.py:391: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



## 해석

**Target 설계:**  
`입결_프록시`를 target으로 둔다. 이 파일은 `P2_G2_정시입결.csv` 단일 CSV에서 온 2024년 대학-학과_계열 단위 데이터이며, `입결_프록시`는 정시 입결 70% cut 또는 정규화된 입결 프록시다.

**관찰:**  
- 입결 반영 행은 3,652개, 커버 대학은 155개다.
- 입결 결측은 6,590개(64.34%)다.
- 입결 0개 대학은 72개다. 이 파일만 보면 원인 분해는 불가능하고, 현상은 대학 단위로 입결 반영 행이 0개인 상태로 확인된다.
- 상관이 큰 변수: 대학원진학률, 전체진학률, 국내진학률, A비율
- 음의 방향 변수: CD비율, F비율, 전문대학진학률
- 입결 중앙값 상위 학과군(n>=10): 의예과, 약, 산업공, 경제, 물리, 소프트웨어

**원인 가설:**  
입결이 높은 행은 진학률과 A비율이 함께 높은 경향이 있고, CD비율은 반대로 움직인다. 다만 이는 학과 구성효과와 대학 단위 선발 구조가 섞인 상관이며 인과로 해석하면 안 된다.

**제한:**  
입결 결측은 무작위 결측이 아니다. 크롤/파싱/학과 매칭 범위에서 생긴 구조적 결측이므로 모델링할 때는 `입결_프록시.notna()`를 명시적으로 필터링해야 한다.

**결론:**  
이 CSV는 입결이 반영된 3,652개 대학-학과 행을 대상으로 성적분포·취업률·진학률과의 탐색 분석에 사용할 수 있다. 다음 개선은 입결 0개 대학과 낮은 커버리지 대학을 줄이는 방향이어야 한다.


saved_dir= /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G2_정시입결_visual_eda
00_source_contract.csv 652
01_missing_profile.csv 771
02_numeric_descriptive_stats.csv 2233
03_correlations_with_admission.csv 1084
04_university_summary.csv 33906
05_department_signal.csv 147490
06_policy_summary.csv 399
07_zero_admission_universities.csv 9046
08_top_bottom_admission_examples.csv 12263
fig01_target_missing_contract.png 170397
fig02_feature_signal_rank.png 133539
fig03_admission_scatter_checked.png 1545947
fig04_university_coverage_checked.png 304396
fig05_group_diagnostics_checked.png 189854


## 산출물

실행 시 `final/eda/P2_G2_정시입결_visual_eda/`에 감사 CSV와 PNG 그래프가 저장된다.